# Product Mapping Quality: experiment

In [1]:
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from pmq import config
from pmq.data import file_sha256, prepare_dataset
from pmq.evaluate import (
    count_group_overlap,
    precision_at_k,
    random_expected_precision,
    rank_by_suspicion,
)
from pmq.model import build_model_text, grouped_splits, out_of_fold_probabilities, suspicion_scores
from pmq.noise import simulate_swaps

print(f"SHA-256: {file_sha256(config.DATA_PATH)}")
print(
    f"SEED={config.SEED}, NOISE_RATE={config.NOISE_RATE}, N_SPLITS={config.N_SPLITS}, "
    f"REVIEW_CAPACITY={config.REVIEW_CAPACITY}"
)

SHA-256: 56f8f699c9e847356666c2eab3c3ab1244340f6a98ad08e39ea2199ebe993ad1
SEED=42, NOISE_RATE=0.1, N_SPLITS=5, REVIEW_CAPACITY=50


## 1. Data

In [2]:
df = prepare_dataset(config.DATA_PATH)
print(f"Offers: {len(df):,}, groups: {df['group_id'].nunique():,}")
df["original_category"].value_counts()

Offers: 5,501, groups: 3,005


original_category
Clothing >> Women's Clothing    3488
Clothing >> Men's Clothing      1517
Clothing >> Kids' Clothing       496
Name: count, dtype: int64

## 2. Simulated errors

In [3]:
simulated = simulate_swaps(df, config.NOISE_RATE, config.SEED)
noise_summary = simulated.groupby("original_category")["is_corrupted"].agg(["size", "sum"])
noise_summary.columns = ["offers", "swapped"]
noise_summary["swapped_percent"] = (noise_summary["swapped"] / noise_summary["offers"] * 100).round(2)
print(f"Swapped: {simulated['is_corrupted'].sum()} ({simulated['is_corrupted'].mean():.2%} of all offers)")
noise_summary

Swapped: 500 (9.09% of all offers)


,offers,swapped,swapped_percent
original_category,,,
Clothing >> Kids' Clothing,496,0,0.00
Clothing >> Men's Clothing,1517,149,9.82
Clothing >> Women's Clothing,3488,351,10.06


## 3. Grouped folds

In [4]:
splits = grouped_splits(simulated["assigned_category"], simulated["group_id"], config.N_SPLITS, config.SEED)
print(
    f"Validation offers whose group is also in training: {count_group_overlap(simulated['group_id'], splits)}"
)
print("Validation sizes:", [len(validation) for _, validation in splits])

Validation offers whose group is also in training: 0
Validation sizes: [1100, 1101, 1100, 1100, 1100]


## 4. Result on simulated errors

In [5]:
texts = build_model_text(simulated)
probabilities = out_of_fold_probabilities(texts, simulated["assigned_category"], splits, config.SEED)
scores = suspicion_scores(probabilities, simulated["assigned_category"])
ranked = rank_by_suspicion(simulated, scores)

k = config.REVIEW_CAPACITY
model_precision = precision_at_k(ranked["is_corrupted"], k)
random_precision = random_expected_precision(simulated["is_corrupted"])
result = pd.DataFrame(
    {
        "swapped_in_top_50": [ranked["is_corrupted"].iloc[:k].sum(), round(k * random_precision, 2)],
        "precision_at_50": [model_precision, round(random_precision, 4)],
    },
    index=["TF-IDF + logistic regression", "random order (expected)"],
)
result

,swapped_in_top_50,precision_at_50
TF-IDF + logistic regression,48.00,0.9600
random order (expected),4.54,0.0909


## 5. Source labels and manual review

In [6]:
source_probabilities = out_of_fold_probabilities(texts, df["original_category"], splits, config.SEED)
source_scores = suspicion_scores(source_probabilities, df["original_category"])
source_ranked = rank_by_suspicion(df, source_scores)
source_ranked["predicted_category"] = source_probabilities.idxmax(axis=1)
top = source_ranked.head(config.REVIEW_CAPACITY)

review = pd.read_csv(config.REVIEW_PATH)
# The verdicts must belong to the same offers in the same order.
assert list(review["pid"]) == list(top["pid"])

print(review["verdict"].value_counts().to_string())
errors = (review["verdict"] == "suspected_mapping_error").sum()
print(f"\nSuspected real mapping errors in top 50: {errors}/{len(review)} = {errors / len(review):.0%}")

verdict
false_alarm                 27
insufficient_information    15
suspected_mapping_error      8

Suspected real mapping errors in top 50: 8/50 = 16%


In [7]:
# Short names for readability.
review["source"] = review["original_category"].str.replace("Clothing >> ", "")
review["model"] = review["predicted_category"].str.replace("Clothing >> ", "")
pd.crosstab([review["source"], review["model"]], review["verdict"])

verdict                            false_alarm  insufficient_information  \
source           model                                                     
Kids' Clothing   Kids' Clothing              2                         0   
                 Men's Clothing              2                         0   
                 Women's Clothing           18                         7   
Men's Clothing   Women's Clothing            5                         5   
Women's Clothing Kids' Clothing              0                         2   
                 Men's Clothing              0                         1   

verdict                            suspected_mapping_error  
source           model                                      
Kids' Clothing   Kids' Clothing                          0  
                 Men's Clothing                          0  
                 Women's Clothing                        2  
Men's Clothing   Women's Clothing                        2  
Women's Clothing Kids' Clothing                          2  
                 Men's Clothing                          2

In [8]:
review[review["verdict"] == "suspected_mapping_error"][["rank", "product_name", "source", "model"]]

,rank,product_name,source,model
0,1,Sharleez Solid Women's A-line Red Skirt,Kids' Clothing,Women's Clothing
1,2,BrandMeUp Solid Women's Straight Kurta,Kids' Clothing,Women's Clothing
2,3,One For Blue Animal Print Women's Round Neck T...,Men's Clothing,Women's Clothing
3,4,TwiQ S.A. Garments Girl's Tube Black Dress,Women's Clothing,Kids' Clothing
4,5,NBA Women's Solid Ankle Length Socks,Men's Clothing,Women's Clothing
5,6,Man2 OSH Kimaka Men's Shapewear,Women's Clothing,Men's Clothing
6,7,Roadster Solid Round Neck Casual Men's Sweater,Women's Clothing,Men's Clothing
10,11,Mee Mee Baby Girl's Textured Stockings,Women's Clothing,Kids' Clothing
